# Principal Component Analysis — From Scratch
> Dimensionality reduction via eigen-decomposition of the covariance matrix.

**Steps:** Center → Covariance → Eigen-decomposition → Project

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_iris
np.random.seed(42)

## PCA Class

In [ ]:
class PCA:
    def __init__(self, n_components):
        self.n_components = n_components
        self.components = None
        self.mean = None
        self.explained_variance_ratio_ = None

    def fit(self, X):
        self.mean = X.mean(axis=0)
        X_c = X - self.mean
        cov = (X_c.T @ X_c) / (len(X) - 1)
        eigenvalues, eigenvectors = np.linalg.eigh(cov)
        # Sort descending
        idx = np.argsort(eigenvalues)[::-1]
        eigenvalues  = eigenvalues[idx]
        eigenvectors = eigenvectors[:, idx]
        self.components = eigenvectors[:, :self.n_components].T
        total = eigenvalues.sum()
        self.explained_variance_ratio_ = eigenvalues / total

    def transform(self, X):
        return (X - self.mean) @ self.components.T

    def fit_transform(self, X):
        self.fit(X); return self.transform(X)

## Iris — 2D Projection

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target

pca = PCA(n_components=2)
X2d = pca.fit_transform(X)

print(f"Variance explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%  "
      f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%")

plt.figure(figsize=(7,5))
for c, name in enumerate(iris.target_names):
    mask = y==c
    plt.scatter(X2d[mask,0], X2d[mask,1], label=name, s=40)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("PCA – Iris 2D Projection")
plt.legend(); plt.tight_layout(); plt.show()

## Scree Plot — Explained Variance

In [ ]:
pca_full = PCA(n_components=4)
pca_full.fit(X)
evr = pca_full.explained_variance_ratio_
cumulative = np.cumsum(evr)

plt.figure(figsize=(7,4))
plt.bar(range(1,5), evr*100, color='steelblue', label='Individual')
plt.step(range(1,5), cumulative*100, where='mid', color='tomato', lw=2, label='Cumulative')
plt.xlabel("Principal Component"); plt.ylabel("Explained Variance (%)")
plt.title("Scree Plot"); plt.legend(); plt.tight_layout(); plt.show()

## Digits — Reconstruct from K Components

In [ ]:
digits = load_digits()
Xd = digits.data  # 1797 × 64

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
sample = Xd[0]
for ax, k in zip(axes.flat, [1, 2, 4, 8, 16, 32, 48, 56, 60, 64]):
    p = PCA(n_components=k); p.fit(Xd)
    recon = p.transform(Xd[:1]) @ p.components + p.mean
    ax.imshow(recon.reshape(8,8), cmap='gray')
    ax.set_title(f"k={k}"); ax.axis('off')
plt.suptitle("Digit Reconstruction at Various k", y=1.01)
plt.tight_layout(); plt.show()